# Module 9: RAG & LLM Applications
## Embeddings: 06 - End-to-End Semantic Search Engine

### Theoretical Overview
Semantic search replaces exact keyword matching with vector proximity ranking.
Workflow:
1. Ingest corpus documents and generate embeddings.
2. Ingest user query and map into the same embedding space.
3. Compute cosine similarity against all document vectors.
4. Return Top-$K$ ranked results with metadata.

In [ ]:
import numpy as np

class InMemorySemanticSearchEngine:
    def __init__(self):
        self.documents = []
        self.vectors = []
        
    def add_documents(self, docs: List[Dict[str, Any]], embeddings: np.ndarray):
        self.documents.extend(docs)
        self.vectors.append(embeddings)
        
    def build_index(self):
        self.matrix = np.vstack(self.vectors)
        # Normalize for fast dot product
        norms = np.linalg.norm(self.matrix, axis=1, keepdims=True)
        self.norm_matrix = self.matrix / np.maximum(norms, 1e-12)
        
    def query(self, query_vector: np.ndarray, top_k: int = 3) -> List[Dict[str, Any]]:
        q_norm = query_vector / max(np.linalg.norm(query_vector), 1e-12)
        similarities = np.dot(self.norm_matrix, q_norm)
        top_indices = np.argsort(similarities)[::-1][:top_k]
        
        results = []
        for idx in top_indices:
            results.append({
                "score": float(similarities[idx]),
                "id": self.documents[idx]["id"],
                "text": self.documents[idx]["text"],
                "category": self.documents[idx]["category"]
            })
        return results

# Test search engine
corpus = [
    {"id": 1, "text": "Python supports object-oriented and functional programming paradigms.", "category": "Programming"},
    {"id": 2, "text": "Deep learning models require GPU hardware for tensor acceleration.", "category": "Hardware/AI"},
    {"id": 3, "text": "Pandas DataFrames provide efficient tabular manipulation tools.", "category": "Data Science"},
    {"id": 4, "text": "ChromaDB and FAISS are popular vector databases for RAG.", "category": "RAG/VectorDB"},
]

np.random.seed(101)
doc_embeddings = np.random.randn(4, 16)
# Make document 4 closely match our query
doc_embeddings[3] += 1.5

engine = InMemorySemanticSearchEngine()
engine.add_documents(corpus, doc_embeddings)
engine.build_index()

mock_query_vec = doc_embeddings[3] + np.random.randn(16) * 0.1
results = engine.query(mock_query_vec, top_k=2)

print("Top Semantic Search Hits:")
for r in results:
    print(f"[Score: {r['score']:.4f}] Doc #{r['id']} ({r['category']}): {r['text']}")